# 01. Ingesta de datos internacionales — TSI

## Objetivo

Implementar y validar la ingesta de los conjuntos de datos
internacionales del proyecto Traffic Stability Index (TSI)
en Databricks.

Se utilizarán archivos Parquet previamente procesados
y limpiados durante la etapa de ingeniería de datos.

## Fuentes

- Estambul
- São Paulo
- Madrid
- METR-LA
- PEMS-BAY

## Estrategia

1. Verificar la disponibilidad de los archivos.
2. Inspeccionar su estructura.
3. Validar el número de observaciones.
4. Convertir los datos a Delta Lake.
5. Comprobar la integridad de los registros.

Los archivos originales no serán modificados.

In [0]:
ruta = "/Volumes/workspace/tsi/datasets/"

for archivo in dbutils.fs.ls(ruta):
    print(archivo.name, archivo.size, "bytes")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DE DATOS DE ESTAMBUL
# ==========================================

RUTA_ISTANBUL = (
    "/Volumes/workspace/tsi/datasets/"
    "istanbul_traffic_index_clean.parquet"
)

# Leer archivo original
df_istanbul = spark.read.parquet(RUTA_ISTANBUL)

# Revisar estructura
print("=== ESTRUCTURA DEL DATASET ===")
df_istanbul.printSchema()

# Mostrar primeros registros
print("\n=== PRIMEROS 5 REGISTROS ===")
df_istanbul.show(5, truncate=False)

# Validar cantidad
total = df_istanbul.count()

print("\n=== VALIDACIÓN DE REGISTROS ===")
print(f"Registros encontrados: {total:,}")
print("Registros esperados: 3,332")

if total == 3332:
    print("VALIDACIÓN CORRECTA")
else:
    print("ADVERTENCIA: revisar cantidad de registros")

## 2. Persistencia en Delta Lake

Se almacena el conjunto limpio de Estambul en una tabla
Delta administrada por Unity Catalog.

La operación conserva las columnas y los registros del
archivo Parquet original, sin aplicar transformaciones
adicionales.

La tabla resultante se utilizará para consultas,
validaciones y posteriores procesos analíticos.

In [0]:
# ==========================================
# TSI - INGESTA DE ESTAMBUL A DELTA LAKE
# Compatible con Databricks Serverless
# ==========================================

TABLA_DESTINO = "workspace.tsi.istanbul_clean"

# Comprobar si la tabla ya existe
if spark.catalog.tableExists(TABLA_DESTINO):
    print(f"La tabla {TABLA_DESTINO} ya existe.")
    print("No se realizó ninguna modificación.")

else:
    (
        df_istanbul.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )

    print("Tabla Delta creada correctamente.")

print(f"Tabla destino: {TABLA_DESTINO}")

In [0]:
# ==========================================
# VALIDACIÓN DE PARQUET VS DELTA
# ==========================================

df_delta = spark.table(
    "workspace.tsi.istanbul_clean"
)

total_parquet = df_istanbul.count()
total_delta = df_delta.count()

print("=== COMPARACIÓN ===")
print(f"Parquet: {total_parquet:,}")
print(f"Delta:   {total_delta:,}")

assert total_parquet == total_delta == 3332

print("\nVALIDACIÓN CORRECTA")
print("Los conteos coinciden.")

print("\n=== FORMATO DE ALMACENAMIENTO ===")

spark.sql("""
    DESCRIBE DETAIL workspace.tsi.istanbul_clean
""").select(
    "format",
    "numFiles",
    "location"
).show(truncate=False)

In [0]:
# ==========================================
# TSI - VALIDACIÓN DE INTEGRIDAD
# PARQUET VS DELTA
# ==========================================

from pyspark.sql import functions as F

# Comparar estructura
assert df_istanbul.schema == df_delta.schema, (
    "Los esquemas no coinciden."
)

# Comparar contenido, incluyendo duplicados
diferencias_origen = (
    df_istanbul.exceptAll(df_delta).count()
)

diferencias_destino = (
    df_delta.exceptAll(df_istanbul).count()
)

print("=== VALIDACIÓN DE INTEGRIDAD ===")
print("Diferencias Parquet → Delta:", diferencias_origen)
print("Diferencias Delta → Parquet:", diferencias_destino)

assert diferencias_origen == 0
assert diferencias_destino == 0

print("\nVALIDACIÓN COMPLETA")
print("Esquema, cantidad y contenido coinciden.")

In [0]:
# ==========================================
# TSI - INSPECCIÓN DE SÃO PAULO
# ==========================================

RUTA_BASE = "/Volumes/workspace/tsi/datasets"

# Identificar el nombre exacto del archivo
for archivo in dbutils.fs.ls(RUTA_BASE):
    print(f"{archivo.name} | {archivo.size:,} bytes")

## 3. Ingesta de São Paulo

Se inspecciona el conjunto de datos limpio de São Paulo
para comprobar su estructura, cantidad de registros y
compatibilidad con Databricks.

La fuente original se conserva sin modificaciones.

In [0]:
# ==========================================
# TSI - VALIDACIÓN SÃO PAULO
# PARQUET COMPATIBLE CON DATABRICKS
# ==========================================

RUTA_SAO_PAULO = (
    "/Volumes/workspace/tsi/datasets/"
    "sao_paulo_cet_clean_databricks.parquet"
)

df_sao_paulo = spark.read.parquet(RUTA_SAO_PAULO)

print("=== ESQUEMA ===")
df_sao_paulo.printSchema()

print("\n=== TOTAL DE REGISTROS ===")
total = df_sao_paulo.count()

print(f"Registros encontrados: {total:,}")
print("Registros esperados: 5,223,411")

assert total == 5_223_411, (
    "ERROR: La cantidad de registros no coincide."
)

print("\nVALIDACIÓN CORRECTA")

In [0]:
# ==========================================
# TSI - CREACIÓN DE TABLA DELTA SÃO PAULO
# ==========================================

TABLA_DESTINO = "workspace.tsi.sao_paulo_clean"

if spark.catalog.tableExists(TABLA_DESTINO):
    print(f"La tabla {TABLA_DESTINO} ya existe.")
    print("No se realizó ninguna modificación.")

else:
    (
        df_sao_paulo.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )

    print("Tabla Delta creada correctamente.")

print(f"Tabla destino: {TABLA_DESTINO}")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DELTA SÃO PAULO
# ==========================================

df_delta_sao_paulo = spark.table(
    "workspace.tsi.sao_paulo_clean"
)

total_parquet = df_sao_paulo.count()
total_delta = df_delta_sao_paulo.count()

print("=== COMPARACIÓN ===")
print(f"Parquet: {total_parquet:,}")
print(f"Delta:   {total_delta:,}")

assert total_parquet == total_delta == 5_223_411

print("\nVALIDACIÓN CORRECTA")
print("Los conteos coinciden.")

print("\n=== FORMATO DE ALMACENAMIENTO ===")

spark.sql(
    "DESCRIBE DETAIL workspace.tsi.sao_paulo_clean"
).select(
    "format",
    "numFiles",
    "location"
).show(truncate=False)

In [0]:
# ==========================================
# TSI - ESTADO ACTUAL DE DATABRICKS
# ==========================================

print("=== ENTORNO ===")
print("Catálogo:", spark.sql("SELECT current_catalog()").first()[0])
print("Esquema:", spark.sql("SELECT current_schema()").first()[0])

print("\n=== TABLAS DEL PROYECTO TSI ===")
spark.sql("SHOW TABLES IN workspace.tsi").show(
    truncate=False
)

print("\n=== ARCHIVOS DEL VOLUMEN ===")
for archivo in dbutils.fs.ls(
    "/Volumes/workspace/tsi/datasets/"
):
    print(f"{archivo.name} | {archivo.size:,} bytes")

print("\n=== REGISTROS EN TABLAS DELTA ===")

tablas = [
    "workspace.tsi.istanbul_clean",
    "workspace.tsi.sao_paulo_clean",
]

for tabla in tablas:
    if spark.catalog.tableExists(tabla):
        total = spark.table(tabla).count()
        print(f"{tabla}: {total:,} registros")
    else:
        print(f"{tabla}: NO EXISTE")

In [0]:
# ==========================================
# TSI - VERIFICACIÓN DE MIGRACIÓN
# ==========================================

TABLAS = {
    "Estambul": {
        "nombre": "workspace.tsi.istanbul_clean",
        "esperado": 3332
    },
    "São Paulo": {
        "nombre": "workspace.tsi.sao_paulo_clean",
        "esperado": 5223411
    }
}

print("=== ESTADO DE TABLAS DELTA ===")

for ciudad, datos in TABLAS.items():

    tabla = datos["nombre"]
    esperado = datos["esperado"]

    print(f"\nFuente: {ciudad}")

    if spark.catalog.tableExists(tabla):

        total = spark.table(tabla).count()

        print(f"Tabla: {tabla}")
        print(f"Registros: {total:,}")
        print(f"Esperados: {esperado:,}")

        if total == esperado:
            print("ESTADO: VALIDADO")
        else:
            print("ESTADO: REQUIERE REVISIÓN")

    else:
        print(f"Tabla: {tabla}")
        print("ESTADO: NO EXISTE")

print("\n=== ARCHIVOS DISPONIBLES ===")

for archivo in dbutils.fs.ls(
    "/Volumes/workspace/tsi/datasets/"
):
    print(f"{archivo.name} | {archivo.size:,} bytes")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DE MADRID
# ==========================================

RUTA_MADRID = (
    "/Volumes/workspace/tsi/datasets/"
    "madrid_traffic_points_clean.parquet"
)

# Leer Parquet
df_madrid = spark.read.parquet(RUTA_MADRID)

print("=== ESQUEMA DE MADRID ===")
df_madrid.printSchema()

print("\n=== TOTAL DE REGISTROS ===")

total_madrid = df_madrid.count()

print(f"Registros encontrados: {total_madrid:,}")
print("Registros esperados: 12,512,133")

assert total_madrid == 12_512_133, (
    "ERROR: El conteo de Madrid no coincide."
)

print("\nVALIDACIÓN CORRECTA")
print("Madrid se leyó correctamente desde Databricks.")

In [0]:
# ==========================================
# TSI - CREACIÓN DE TABLA DELTA MADRID
# ==========================================

TABLA_MADRID = "workspace.tsi.madrid_clean"

if spark.catalog.tableExists(TABLA_MADRID):
    print(f"La tabla {TABLA_MADRID} ya existe.")
    print("No se realizaron modificaciones.")
else:
    (
        df_madrid.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_MADRID)
    )

    print("Tabla Delta de Madrid creada correctamente.")

print(f"Tabla destino: {TABLA_MADRID}")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DELTA MADRID
# ==========================================

df_delta_madrid = spark.table(
    "workspace.tsi.madrid_clean"
)

total_parquet = df_madrid.count()
total_delta = df_delta_madrid.count()

print("=== COMPARACIÓN ===")
print(f"Parquet: {total_parquet:,}")
print(f"Delta:   {total_delta:,}")

assert total_parquet == total_delta == 12_512_133

print("\nVALIDACIÓN CORRECTA")
print("Los conteos coinciden.")

print("\n=== FORMATO DE ALMACENAMIENTO ===")

spark.sql(
    "DESCRIBE DETAIL workspace.tsi.madrid_clean"
).select(
    "format",
    "numFiles"
).show(truncate=False)

In [0]:
# ==========================================
# TSI - INTEGRIDAD PARQUET VS DELTA
# MADRID
# ==========================================

print("=== VALIDACIÓN DE ESQUEMA ===")

assert df_madrid.schema == df_delta_madrid.schema, (
    "ERROR: Los esquemas no coinciden."
)

print("Esquemas idénticos.")

print("\n=== VALIDACIÓN DE CONTENIDO ===")

diferencias_origen = (
    df_madrid.exceptAll(df_delta_madrid).count()
)

diferencias_destino = (
    df_delta_madrid.exceptAll(df_madrid).count()
)

print(f"Parquet → Delta: {diferencias_origen:,}")
print(f"Delta → Parquet: {diferencias_destino:,}")

assert diferencias_origen == 0
assert diferencias_destino == 0

print("\nVALIDACIÓN COMPLETA")
print("Madrid conserva el esquema y todos los registros.")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DE METR-LA
# ==========================================

RUTA_BASE = "/Volumes/workspace/tsi/datasets/"

# Identificar el archivo subido
archivos_metr = [
    archivo.name
    for archivo in dbutils.fs.ls(RUTA_BASE)
    if archivo.name.lower().endswith(".parquet")
    and "metr" in archivo.name.lower()
]

print("=== ARCHIVOS METR-LA ===")
print(archivos_metr)

assert len(archivos_metr) == 1, (
    "Se esperaba encontrar exactamente un Parquet de METR-LA."
)

RUTA_METR = RUTA_BASE + archivos_metr[0]

# Leer el archivo original
df_metr = spark.read.parquet(RUTA_METR)

print("\n=== ESQUEMA ===")
df_metr.printSchema()

print("\n=== TOTAL DE REGISTROS ===")
total_metr = df_metr.count()

print(f"Registros encontrados: {total_metr:,}")
print("Registros esperados: 7,094,304")

assert total_metr == 7_094_304, (
    "ERROR: El conteo de METR-LA no coincide."
)

print("\nVALIDACIÓN CORRECTA")
print("METR-LA se leyó correctamente.")

In [0]:
# ==========================================
# TSI - CREACIÓN DE TABLA DELTA METR-LA
# ==========================================

TABLA_METR = "workspace.tsi.metr_la_clean"

if spark.catalog.tableExists(TABLA_METR):
    print(f"La tabla {TABLA_METR} ya existe.")
    print("No se realizaron modificaciones.")

else:
    (
        df_metr.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_METR)
    )

    print("Tabla Delta de METR-LA creada correctamente.")

print(f"Tabla destino: {TABLA_METR}")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DELTA METR-LA
# ==========================================

df_delta_metr = spark.table(
    "workspace.tsi.metr_la_clean"
)

total_parquet = df_metr.count()
total_delta = df_delta_metr.count()

print("=== COMPARACIÓN ===")
print(f"Parquet: {total_parquet:,}")
print(f"Delta:   {total_delta:,}")

assert total_parquet == total_delta == 7_094_304, (
    "ERROR: Los conteos no coinciden."
)

print("\nVALIDACIÓN CORRECTA")
print("Los conteos coinciden.")

print("\n=== FORMATO DE ALMACENAMIENTO ===")

spark.sql(
    "DESCRIBE DETAIL workspace.tsi.metr_la_clean"
).select(
    "format",
    "numFiles"
).show(truncate=False)

In [0]:
# ==========================================
# TSI - INTEGRIDAD PARQUET VS DELTA
# METR-LA
# ==========================================

print("=== VALIDACIÓN DE ESQUEMA ===")

assert df_metr.schema == df_delta_metr.schema, (
    "ERROR: Los esquemas no coinciden."
)

print("Esquemas idénticos.")

print("\n=== VALIDACIÓN DE CONTENIDO ===")

diferencias_origen = (
    df_metr.exceptAll(df_delta_metr).count()
)

diferencias_destino = (
    df_delta_metr.exceptAll(df_metr).count()
)

print(f"Parquet → Delta: {diferencias_origen:,}")
print(f"Delta → Parquet: {diferencias_destino:,}")

assert diferencias_origen == 0
assert diferencias_destino == 0

print("\nVALIDACIÓN COMPLETA")
print("METR-LA conserva el esquema y todos los registros.")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DE PEMS-BAY
# ==========================================

RUTA_PEMS = (
    "/Volumes/workspace/tsi/datasets/"
    "pems_bay_clean.parquet"
)

df_pems = spark.read.parquet(RUTA_PEMS)

print("=== ESQUEMA PEMS-BAY ===")
df_pems.printSchema()

print("\n=== TOTAL DE REGISTROS ===")

total_pems = df_pems.count()

print(f"Registros encontrados: {total_pems:,}")
print("Registros esperados: 16,937,700")

assert total_pems == 16_937_700, (
    "ERROR: El conteo de PEMS-BAY no coincide."
)

print("\nVALIDACIÓN CORRECTA")

In [0]:
# ==========================================
# TSI - CREACIÓN DE TABLA DELTA PEMS-BAY
# ==========================================

TABLA_PEMS = "workspace.tsi.pems_bay_clean"

if spark.catalog.tableExists(TABLA_PEMS):
    print(f"La tabla {TABLA_PEMS} ya existe.")
    print("No se realizaron modificaciones.")

else:
    (
        df_pems.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_PEMS)
    )

    print("Tabla Delta de PEMS-BAY creada correctamente.")

print(f"Tabla destino: {TABLA_PEMS}")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DELTA PEMS-BAY
# ==========================================

df_delta_pems = spark.table(
    "workspace.tsi.pems_bay_clean"
)

total_parquet = df_pems.count()
total_delta = df_delta_pems.count()

print("=== COMPARACIÓN ===")
print(f"Parquet: {total_parquet:,}")
print(f"Delta:   {total_delta:,}")

assert total_parquet == total_delta == 16_937_700, (
    "ERROR: Los conteos no coinciden."
)

print("\nVALIDACIÓN CORRECTA")
print("Los conteos coinciden.")

print("\n=== FORMATO DE ALMACENAMIENTO ===")

spark.sql(
    "DESCRIBE DETAIL workspace.tsi.pems_bay_clean"
).select(
    "format",
    "numFiles"
).show(truncate=False)

In [0]:
# ==========================================
# TSI - INTEGRIDAD PARQUET VS DELTA
# PEMS-BAY
# ==========================================

print("=== VALIDACIÓN DE ESQUEMA ===")

assert df_pems.schema == df_delta_pems.schema, (
    "ERROR: Los esquemas no coinciden."
)

print("Esquemas idénticos.")

print("\n=== VALIDACIÓN DE CONTENIDO ===")

diferencias_origen = (
    df_pems.exceptAll(df_delta_pems).count()
)

diferencias_destino = (
    df_delta_pems.exceptAll(df_pems).count()
)

print(f"Parquet → Delta: {diferencias_origen:,}")
print(f"Delta → Parquet: {diferencias_destino:,}")

assert diferencias_origen == 0
assert diferencias_destino == 0

print("\nVALIDACIÓN COMPLETA")
print("PEMS-BAY conserva el esquema y todos los registros.")

In [0]:
# ==========================================
# TSI - INTEGRIDAD PARQUET VS DELTA
# SÃO PAULO
# ==========================================

RUTA_SAO_PAULO = (
    "/Volumes/workspace/tsi/datasets/"
    "sao_paulo_cet_clean_databricks.parquet"
)

df_sao_paulo = spark.read.parquet(RUTA_SAO_PAULO)

df_delta_sao_paulo = spark.table(
    "workspace.tsi.sao_paulo_clean"
)

print("=== VALIDACIÓN DE ESQUEMA ===")

assert df_sao_paulo.schema == df_delta_sao_paulo.schema, (
    "ERROR: Los esquemas no coinciden."
)

print("Esquemas idénticos.")

print("\n=== VALIDACIÓN DE CONTENIDO ===")

diferencias_origen = (
    df_sao_paulo.exceptAll(df_delta_sao_paulo).count()
)

diferencias_destino = (
    df_delta_sao_paulo.exceptAll(df_sao_paulo).count()
)

print(f"Parquet → Delta: {diferencias_origen:,}")
print(f"Delta → Parquet: {diferencias_destino:,}")

assert diferencias_origen == 0
assert diferencias_destino == 0

print("\nVALIDACIÓN COMPLETA")
print("São Paulo conserva el esquema y todos los registros.")

In [0]:
# ==========================================
# TSI - VALIDACIÓN GLOBAL DELTA LAKE
# ==========================================

TABLAS_TSI = {
    "Estambul": ("workspace.tsi.istanbul_clean", 3332),
    "São Paulo": ("workspace.tsi.sao_paulo_clean", 5223411),
    "Madrid": ("workspace.tsi.madrid_clean", 12512133),
    "METR-LA": ("workspace.tsi.metr_la_clean", 7094304),
    "PEMS-BAY": ("workspace.tsi.pems_bay_clean", 16937700),
}

TOTAL_ESPERADO = 41_770_880
total_global = 0

print("=== VALIDACIÓN GLOBAL TSI ===")

for ciudad, (tabla, esperado) in TABLAS_TSI.items():

    assert spark.catalog.tableExists(tabla), (
        f"No existe la tabla: {tabla}"
    )

    total = spark.table(tabla).count()

    formato = (
        spark.sql(f"DESCRIBE DETAIL {tabla}")
        .select("format")
        .first()[0]
    )

    assert total == esperado, (
        f"Conteo incorrecto en {ciudad}"
    )

    assert formato == "delta", (
        f"Formato incorrecto en {ciudad}"
    )

    total_global += total

    print(
        f"{ciudad}: {total:,} registros "
        f"| formato: {formato} | OK"
    )

print("\n=== RESULTADO GLOBAL ===")

print(f"Total encontrado: {total_global:,}")
print(f"Total esperado:   {TOTAL_ESPERADO:,}")

assert total_global == TOTAL_ESPERADO, (
    "ERROR: El total internacional no coincide."
)

print("\nVALIDACIÓN GLOBAL CORRECTA")
print("Las cinco fuentes están completas en Delta Lake.")